# Phishing Email Classification using Machine Learning

This notebook analyzes and classifies emails as phishing or legitimate using various ML techniques.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Text preprocessing
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

# Deep Learning
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, GlobalMaxPooling1D
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Download nltk data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

## 2. Load and Explore Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('phishing_legit_dataset_KD_10000.csv')

# Display basic info
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Check for missing values
print("Missing Values:")
print(df.isnull().sum())

In [ ]:
# Dataset info
df.info()

In [ ]:
# Target variable distribution
print("Label Distribution:")
print(df['label'].value_counts())
print("\nPercentage:")
print(df['label'].value_counts(normalize=True) * 100)

In [ ]:
# Visualize label distribution
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='label')
plt.title('Distribution of Labels (0: Legitimate, 1: Phishing)')
plt.xlabel('Label')
plt.ylabel('Count')
plt.show()

In [ ]:
# Check phishing types
print("Phishing Type Distribution:")
print(df[df['label']==1]['phishing_type'].value_counts())

In [ ]:
# Check severity distribution
print("Severity Distribution:")
print(df['severity'].value_counts())

In [ ]:
# Check confidence scores
print("Confidence Score Statistics:")
print(df['confidence'].describe())

In [ ]:
# Visualize confidence scores by label
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='label', y='confidence')
plt.title('Confidence Scores by Label')
plt.xlabel('Label (0: Legitimate, 1: Phishing)')
plt.ylabel('Confidence Score')
plt.show()

## 3. Text Preprocessing

In [ ]:
# Create a copy for preprocessing
df_clean = df.copy()

# Define stopwords
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

def preprocess_text(text):
    """
    Preprocess text: lowercasing, remove punctuation, remove stopwords, stemming
    """
    if isinstance(text, str):
        # Lowercase
        text = text.lower()
        # Remove punctuation
        text = re.sub(r'[^\w\s]', ' ', text)
        # Remove numbers
        text = re.sub(r'\d+', ' ', text)
        # Remove extra whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        # Tokenize
        tokens = word_tokenize(text)
        # Remove stopwords and stem
        tokens = [stemmer.stem(token) for token in tokens if token not in stop_words and len(token) > 2]
        return ' '.join(tokens)
    return ''

# Apply preprocessing
df_clean['clean_text'] = df_clean['text'].apply(preprocess_text)

# Show sample
print("Original:")
print(df_clean['text'][0])
print("\nPreprocessed:")
print(df_clean['clean_text'][0])

In [ ]:
# Calculate text length statistics
df_clean['text_length'] = df_clean['clean_text'].apply(lambda x: len(x.split()))
print("Text Length Statistics:")
print(df_clean['text_length'].describe())

In [ ]:
# Visualize text length by label
plt.figure(figsize=(10, 5))
sns.histplot(data=df_clean, x='text_length', hue='label', bins=50, alpha=0.6)
plt.title('Text Length Distribution by Label')
plt.xlabel('Number of Words')
plt.ylabel('Count')
plt.show()

## 4. Exploratory Data Analysis

In [ ]:
# Most common words in legitimate vs phishing emails
def get_top_ngrams(corpus, n=20, ngram_range=(1, 1)):
    vectorizer = CountVectorizer(ngram_range=ngram_range, max_features=n)
    X = vectorizer.fit_transform(corpus)
    words = vectorizer.get_feature_names_out()
    counts = X.sum(axis=0).A1
    return pd.DataFrame({'word': words, 'count': counts}).sort_values('count', ascending=False)

# Separate legitimate and phishing
legit_corpus = df_clean[df_clean['label'] == 0]['clean_text']
phish_corpus = df_clean[df_clean['label'] == 1]['clean_text']

# Get top words
top_legit = get_top_ngrams(legit_corpus, 20)
top_phish = get_top_ngrams(phish_corpus, 20)

print("Top 10 words in Legitimate Emails:")
print(top_legit.head(10))

print("\nTop 10 words in Phishing Emails:")
print(top_phish.head(10))

In [ ]:
# Visualize top words
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=top_legit.head(10), x='count', y='word', ax=axes[0])
axes[0].set_title('Top Words in Legitimate Emails')
sns.barplot(data=top_phish.head(10), x='count', y='word', ax=axes[1])
axes[1].set_title('Top Words in Phishing Emails')
plt.tight_layout()
plt.show()

## 5. Feature Engineering

In [ ]:
# Additional features
def extract_features(text):
    features = {}
    # Number of words
    features['word_count'] = len(text.split())
    # Number of characters
    features['char_count'] = len(text)
    # Number of uppercase words (indicators of urgency)
    features['upper_count'] = len(re.findall(r'[A-Z]', text))
    # Number of exclamation marks
    features['exclamation_count'] = text.count('!')
    # Number of question marks
    features['question_count'] = text.count('?')
    # Number of special characters
    features['special_count'] = len(re.findall(r'[^a-zA-Z0-9\s]', text))
    return features

# Apply feature extraction
feature_df = pd.DataFrame(df_clean['text'].apply(extract_features).tolist())
df_clean = pd.concat([df_clean, feature_df], axis=1)

# Show feature statistics
print("Feature Statistics:")
print(df_clean[['word_count', 'char_count', 'upper_count', 'exclamation_count', 'question_count']].describe())

In [ ]:
# Visualize feature distributions by label
features_to_plot = ['word_count', 'char_count', 'upper_count', 'exclamation_count']
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for idx, feature in enumerate(features_to_plot):
    row = idx // 2
    col = idx % 2
    sns.boxplot(data=df_clean, x='label', y=feature, ax=axes[row, col])
    axes[row, col].set_title(f'{feature} by Label')
plt.tight_layout()
plt.show()

## 6. Text Vectorization

In [ ]:
# Prepare data for modeling
X_text = df_clean['clean_text']
X_features = df_clean[['word_count', 'char_count', 'upper_count', 'exclamation_count', 'question_count']]
y = df_clean['label']

# Train-test split
X_train_text, X_test_text, X_train_feat, X_test_feat, y_train, y_test = train_test_split(
    X_text, X_features, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set size: {len(y_train)}")
print(f"Test set size: {len(y_test)}")

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train_text)
X_test_tfidf = tfidf.transform(X_test_text)

print(f"TF-IDF Shape: {X_train_tfidf.shape}")

In [ ]:
# Combine TF-IDF with additional features
from scipy.sparse import hstack

X_train_combined = hstack([X_train_tfidf, X_train_feat.values])
X_test_combined = hstack([X_test_tfidf, X_test_feat.values])

print(f"Combined features shape: {X_train_combined.shape}")

## 7. Traditional Machine Learning Models

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Naive Bayes': MultinomialNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# Train and evaluate models
results = {}
for name, model in models.items():
    model.fit(X_train_combined, y_train)
    y_pred = model.predict(X_test_combined)
    y_pred_proba = model.predict_proba(X_test_combined)[:, 1]
    
    accuracy = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba)
    
    results[name] = {
        'accuracy': accuracy,
        'auc': auc,
        'model': model,
        'y_pred': y_pred,
        'y_pred_proba': y_pred_proba
    }
    
    print(f"{name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  AUC: {auc:.4f}")
    print()

In [ ]:
# Compare results
results_df = pd.DataFrame({
    'Model': list(results.keys()),
    'Accuracy': [results[m]['accuracy'] for m in results],
    'AUC': [results[m]['auc'] for m in results]
})
print("Model Comparison:")
print(results_df)

In [ ]:
# Visualize results
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=results_df, x='Model', y='Accuracy', ax=axes[0])
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)

sns.barplot(data=results_df, x='Model', y='AUC', ax=axes[1])
axes[1].set_title('Model AUC Comparison')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## 8. Confusion Matrices

In [ ]:
# Plot confusion matrices
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for idx, (name, result) in enumerate(results.items()):
    row = idx // 2
    col = idx % 2
    cm = confusion_matrix(y_test, result['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[row, col])
    axes[row, col].set_title(f'{name}')
    axes[row, col].set_xlabel('Predicted')
    axes[row, col].set_ylabel('Actual')
plt.tight_layout()
plt.show()

In [ ]:
# Print classification reports
for name, result in results.items():
    print(f"\n{name} Classification Report:")
    print(classification_report(y_test, result['y_pred'], target_names=['Legitimate', 'Phishing']))

## 9. Best Model Analysis

In [ ]:
# Find best model based on accuracy
best_model_name = results_df.loc[results_df['Accuracy'].idxmax(), 'Model']
best_model = results[best_model_name]['model']

print(f"Best Model: {best_model_name}")
print(f"Accuracy: {results[best_model_name]['accuracy']:.4f}")
print(f"AUC: {results[best_model_name]['auc']:.4f}")

In [ ]:
# Feature importance for Random Forest (if selected)
if best_model_name == 'Random Forest':
    feature_importance = best_model.feature_importances_
    # Get feature names
    feature_names = list(tfidf.get_feature_names_out()) + list(X_features.columns)
    
    # Top features
    importance_df = pd.DataFrame({
        'feature': feature_names,
        'importance': feature_importance
    }).sort_values('importance', ascending=False)
    
    print("Top 20 most important features:")
    print(importance_df.head(20))

In [ ]:
# ROC Curve comparison
plt.figure(figsize=(10, 8))
for name, result in results.items():
    fpr, tpr, _ = roc_curve(y_test, result['y_pred_proba'])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {result["auc"]:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.grid(True)
plt.show()

## 10. Cross-Validation

In [ ]:
# Perform cross-validation for the best model
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(best_model, X_train_combined, y_train, cv=5, scoring='accuracy')
print(f"5-Fold Cross-Validation Scores: {cv_scores}")
print(f"Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 11. Deep Learning Model (LSTM)

In [ ]:
# Prepare text for LSTM
max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

print(f"LSTM Input Shape: {X_train_pad.shape}")

In [ ]:
# Build LSTM model
lstm_model = Sequential([
    Embedding(max_words, 128, input_length=max_len),
    Bidirectional(LSTM(64, return_sequences=True)),
    GlobalMaxPooling1D(),
    Dropout(0.5),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

lstm_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

In [ ]:
# Train LSTM
history = lstm_model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test_pad, y_test),
    verbose=1
)

In [ ]:
# Evaluate LSTM
lstm_pred_proba = lstm_model.predict(X_test_pad)
lstm_pred = (lstm_pred_proba > 0.5).astype(int)

lstm_accuracy = accuracy_score(y_test, lstm_pred)
lstm_auc = roc_auc_score(y_test, lstm_pred_proba)

print(f"LSTM Model:")
print(f"  Accuracy: {lstm_accuracy:.4f}")
print(f"  AUC: {lstm_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, lstm_pred, target_names=['Legitimate', 'Phishing']))

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
plt.tight_layout()
plt.show()

## 12. Final Comparison

In [ ]:
# Compare all models including LSTM
all_results = results_df.copy()
all_results.loc[len(all_results)] = ['LSTM', lstm_accuracy, lstm_auc]
print("Final Model Comparison:")
print(all_results)

In [ ]:
# Visualize final comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.barplot(data=all_results, x='Model', y='Accuracy', ax=axes[0])
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)

sns.barplot(data=all_results, x='Model', y='AUC', ax=axes[1])
axes[1].set_title('Model AUC Comparison')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

In [ ]:
# Add confusion matrix for LSTM
cm_lstm = confusion_matrix(y_test, lstm_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Blues')
plt.title('LSTM Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 13. Conclusion

In [ ]:
print("=" * 60)
print("PHISHING DETECTION - FINAL SUMMARY")
print("=" * 60)

print(f"\nDataset Size: {len(df)} emails")
print(f"Legitimate: {len(df[df['label']==0])}")
print(f"Phishing: {len(df[df['label']==1])}")

print(f"\nBest Traditional Model: {best_model_name}")
print(f"Accuracy: {results[best_model_name]['accuracy']*100:.2f}%")
print(f"AUC: {results[best_model_name]['auc']*100:.2f}%")

print(f"\nLSTM Model Results:")
print(f"Accuracy: {lstm_accuracy*100:.2f}%")
print(f"AUC: {lstm_auc*100:.2f}%")

print("\n" + "=" * 60)
print("Key Insights:")
print("1. Urgency keywords (act now, urgent, expires) are strong phishing indicators")
print("2. Authority scams and credential harvesting are the most common phishing types")
print("3. Text length and special character counts help distinguish phishing emails")
print("4. Both traditional ML and LSTM perform well on this dataset")
print("=" * 60)

## 14. Save Models (Optional)

In [ ]:
# Save the best model
import joblib

# Save best traditional model
joblib.dump(best_model, 'best_phishing_model.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')

# Save LSTM model
lstm_model.save('lstm_phishing_model.h5')

print("Models saved successfully!")

## 15. Prediction Function

In [ ]:
def predict_phishing(text, model, tfidf_vec, feature_func):
    """
    Predict if an email is phishing or legitimate
    """
    # Preprocess text
    clean_text = preprocess_text(text)
    
    # Extract features
    features = feature_func(text)
    feature_values = [features['word_count'], features['char_count'], features['upper_count'], 
                     features['exclamation_count'], features['question_count']]
    
    # TF-IDF vectorize
    tfidf_features = tfidf_vec.transform([clean_text])
    
    # Combine features
    from scipy.sparse import hstack
    combined = hstack([tfidf_features, np.array(feature_values).reshape(1, -1)])
    
    # Predict
    pred = model.predict(combined)[0]
    prob = model.predict_proba(combined)[0][1]
    
    return pred, prob

# Test the prediction function
test_email = "Your account has been compromised. Click here to verify your identity immediately or your account will be suspended."
pred, prob = predict_phishing(test_email, best_model, tfidf, extract_features)
print(f"Prediction: {'Phishing' if pred == 1 else 'Legitimate'}")
print(f"Confidence: {prob*100:.2f}%")